# Assignment 3: LLMs and Machine Learning

---

## Submission Instructions

Submit only a link to the folder for Assignment 3 in your personal GitHub repository. Within the repository, you should have a Jupyter notebook file titled e.g. `assignment3.ipynb` or something similar, placed inside the `assignments/assignment3/` folder.

Make sure the repository is public.

**Submissions must be made using a GitHub repository. Submissions that do not follow this instruction will receive 0 points.**

**Late submissions are not accepted as the peer review system does not allow adding submissions past the deadline. Submit your work early to not miss the deadline!**

## Code Quality

Write your code so that it is pleasant to read and easy to understand. This includes:

- Use descriptive variable and function names.
- Add brief comments where the logic is not immediately obvious.
- Keep your notebook organized with clear separation between tasks.
- Print out your answers so that the peer reviewer can see the results. Use the `df.head()` when asked to print the top  5 lines. To print a better looking DataFrame, consider also using `display()` instead of `print()`.
- Divide the code into logical chunks. At minimum, use separate cells per task, and when reasonable, separate cells for subtasks.
- Remember to in the end rerun all code from the beginning to end of the notebook to ensure each cell runs without error

## Visualizations

In the visualizations always include enough information that the plot can be understood independently. This includes:

- Labels for both axes
- A descriptive title

## Statement of use of AI

Include a brief statement describing how and which AI was used (or if no AI was used) in completing the assignment. This could be a markdown cell with a couple of sentences. As a reminder, AI use is permitted in the assignments, but it is advisable to try to complete the tasks as far as possible without and to make sure you understand the code that AI produced when using it.

## Grading

This assignment is worth 10 points. Task 0 is worth 1 point, and tasks 1-2 are worth 2 points and task 3 is worth 5 points.

Points are given only for code that runs. If the code does not run, the task (or subtask if code for a task is divided into multiple cells) will automatically receive 0 points even if the code is almost correct.

### Penalties

- **-2 points per task** where AI-generated (hallucinated) data is used instead of the actual data provided in the task or retrieved from the specified source. The assignment requires working with real data, not made-up values!
- **-3 points** if an API key is included in the submission notebook or anywhere in the GitHub repository. Store your keys in a `.env` file and add `.env` to your `.gitignore`.
- **-1 point** if the Jupyter Notebook is overall messy and not structured well (e.g. if all tasks are completed within one cell, if answers are difficult to find due to too much irrelevant printed output).
- **-1 point** if there is no statement of AI use. If you did not use AI, report that you did not use AI.

### Editing the submission after the deadline

- Editing the assignment submission during the evaluation phase is forbidden. Thus, after the solution has been released, do not make any further changes to the notebook until you have received a grade. If you accidentally leaked an API key, revoke the key immediately. Other **changes to the submission are considered cheating, and will result in 0 points for both the assignment and peer review**.

---

## Tasks

### Task 0: Setting up Ollama (1p)

a) Set up Ollama and connect to it using either openAI's API or Ollama's own API. 

b) Load the 270m parameter version of the [gemma3](https://ollama.com/library/gemma3) model and test it with any prompt.

c) Load the 4b parameter version of the [gemma3](https://ollama.com/library/gemma3) and test it with any prompt. If running the 4b version is too slow, you can use the 1b version instead.

In [201]:
# Task A 

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama"
)

In [203]:
# Task B
# I first ran this command in my Mac terminal: ollama run gemma3:270m

def ask_llm_270m(prompt):
    response = client.chat.completions.create(
        model="gemma3:270m",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print(ask_llm_270m("In one sentence, what is the capital of Finland?"))

# This model is very weak, often hallusinating.

The capital of Finland is Helsinki.



In [204]:
# Task C
# I first ran this command in my Mac terminal: ollama run gemma3:4b

def ask_llm_4b(prompt):
    response = client.chat.completions.create(
        model="gemma3:4b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print(ask_llm_4b("in one sentence, explain to what category of the Köppen climate classification does Finland belong to"))

Finland primarily belongs to the **Dfc (temperate with hot summer) climate** category within the Köppen classification, characterized by warm summers and cold, wet winters.


### Task 1: Text classification with Ollama (2p)

The `data/emails.csv` file contains 12 email headlines, with 4 spam emails, 4 legitimate work emails and 4 vague emails that are hard to classify based on the title alone. Use this dataset for all subtasks in this task.

a) Make a function for classifying emails (based on the headlines) as spam, work or unknown. The function should return only the classification and nothing else. (0.5p)

b) Use the smaller gemma3 (270m) to classify the emails using the function created in part a. (0.5p)

c) Use larger gemma3 (4b) to classify the emails using the function created in part a). In separate markdown cell, write a brief comment comparing the results of parts b) and c). (0.5p)

d) Write a script that repeats b) and c) 3 times, storing the results for both models separately. For both models, put the results as columns into a new DataFrame that also contains the headlines so that it is easy to compare how the output varied across runs for both models. Comment if there were differences and explain why this happened. (0.5p)

In [249]:
# Task A

import pandas as pd

emails = pd.read_csv("data/emails.csv", sep=";")

def classify_emails(emails_to_classify, model_name):
    prompt = f"""
Classify each email headline as exactly one of these labels:
spam, work, unknown

Return only the classifications and no extra explanation or text.

Email headlines:
{emails_to_classify}
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()


In [250]:
# Task B
print(classify_emails(emails, "gemma3:270m"))
# The answer with this 270m model is often pure hallucination

1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12


In [251]:
# Task C
print(classify_emails(emails, "gemma3:4b"))
# The answer with this 4b model is often much better

0: spam
1: spam
2: spam
3: work
4: work
5: work
6: work
7: work
8: work
9: work
10: work
11: unknown


### Task c summary note
The 270m model responses are incoherent, while the 4b model responses are coherent

In [252]:
# Task d

def response_to_rows(output, expected_length):
    labels = output.splitlines()
    labels = [label.strip().lower() for label in labels if label.strip() != ""]

    if len(labels) == expected_length:
        return labels
    else:
        return ["incoherent answer"] * expected_length


comparison = emails.copy()
expected_length = len(emails)

for i in range(1, 4):
    output_4b = classify_emails(emails, "gemma3:4b")
    comparison[f"gemma3:4b run {i}"] = response_to_rows(output_4b, expected_length)

for i in range(1, 4):
    output_270m = classify_emails(emails, "gemma3:270m")
    comparison[f"gemma3:270m run {i}"] = response_to_rows(output_270m, expected_length)

display(comparison)

,headline,gemma3:4b run 1,gemma3:4b run 2,gemma3:4b run 3,gemma3:270m run 1,gemma3:270m run 2,gemma3:270m run 3
0,URGENT: Your account will be suspended within ...,0: spam,0 spam,0: spam,email headlines:,incoherent answer,incoherent answer
1,Congratulations! You have won a 1000€ gift car...,1: spam,1 spam,1: spam,1. urgent: your account will be suspended wit...,incoherent answer,incoherent answer
2,Hot singles in your area are waiting to meet y...,2: spam,2 spam,2: spam,2. congratulations! you have won a 1000€ gift...,incoherent answer,incoherent answer
3,Re: Inheritance transfer of 4.5M USD pending y...,3: work,3 work,3: work,3. re: inheritance transfer of 4.5m usd pendi...,incoherent answer,incoherent answer
4,Meeting agenda for Thursday's project review,4: work,4 work,4: work,4. meeting agenda for thursday's project re...,incoherent answer,incoherent answer
5,"Q3 budget report attached, please review by Fr...",5: work,5 work,5: work,"5. q3 budget report attached, please review b...",incoherent answer,incoherent answer
6,Reminder: Annual performance review scheduled ...,6: work,6 work,6: work,6. reminder: annual performance review schedu...,incoherent answer,incoherent answer
7,"Updated draft of the manuscript, comments welcome",7: work,7 work,7: work,"7. updated draft of the manuscript, comments ...",incoherent answer,incoherent answer
8,Quick question about last week,8: work,8 unknown,8: work,8. quick question about last week,incoherent answer,incoherent answer
9,Following up,9: work,9 unknown,9: work,9. follow up,incoherent answer,incoherent answer


### Summary

There were clear differences between the two models. The `gemma3:4b` model usually followed the instructions well and returned 12 classifications, so the answers could be placed correctly into the DataFrame rows. In contrast, every output I received from the `gemma3:270m` model was incoherent or incorrect for this task, so none of its runs could be used as proper row-by-row classifications. I also had a hard time making sure the results did not have numbers infront. Askin the model to not include numbers infront labels sometimes crashed the results that the AI returned.

If a model did not return exactly 12 rows, I marked the whole run as `incoherent answer`. This was done because it was not possible to know which email headline the model had skipped or answered incorrectly. Placing those answers into the DataFrame row by row could therefore have matched classifications to the wrong emails and made the comparison misleading.

The difference likely happened because `gemma3:270m` is a much smaller model than `gemma3:4b`, so it has weaker instruction-following ability and could not handle the complexity of this task reliably.

### Task 2: Sentiment analysis with Ollama (2p)

The `data/news.csv` file contains 10 fictional financial news headlines. Use it for all subtasks in this task.

a) Make a function for classifying the texts in the provided dataset based on the topic (earnings, mergers, regulation, macroeconomics) and for determining the sentiment of the news (positive, negative, neutral). The function should return the class and sentiment in JSON format. (1p)

b) Use gemma3 (4b) to classify and provide the sentiment for each row of the provided dataset, inserting them into a new DataFrame that contains both the original headlines as well as topic and sentiment. (0.5p)

c) Give the same data and prompt to a browser based LLM (e.g. ChatGPT, LeChat, Claude or Gemini) and ask it to provide the topic and sentiment, giving it the same options. Paste the results into a markdown cell. Compare the results of b) and c), which one is more accurate and why? (0.5p)

In [210]:
# Task A

news = pd.read_csv("data/news.csv", sep=":")
news.head()

def classify_text(dataset, model):
    prompt = f"""Read through the news headlines and categorize them according to:
topic: earnings, mergers, regulation, macroeconomics
sentiment: positive, negative, neutral

Return only a JSON list.
Do not use markdown.
Do not use ```json.
Do not include explanations.
News = {dataset}
"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content.strip()

In [211]:
# Task B
from io import StringIO
response = pd.read_json(StringIO(classify_text(news, "gemma3:4b")))

summary_gemma = pd.DataFrame()
summary_gemma["headline"] = news["headline"]
summary_gemma["topic"] = response["topic"]
summary_gemma["sentiment"] = response["sentiment"]

display(summary_gemma)

,headline,topic,sentiment
0,Nordion Industries beats Q1 earnings estimates...,earnings,positive
1,Helvora Pharmaceuticals misses earnings foreca...,earnings,negative
2,"Aurelis Bank reports steady quarterly profit, ...",earnings,positive
3,Veridyne Logistics to acquire rival Trantec in...,mergers,neutral
4,Antitrust regulators block proposed merger bet...,regulation,negative
5,Kestrel Semiconductor confirms early-stage mer...,mergers,neutral
6,New EU AI Act compliance rules expected to rai...,regulation,negative
7,Finnish FSA grants Norvik Capital expanded lic...,regulation,positive
8,"Eurozone inflation cools to 2.1%, easing press...",macroeconomics,positive
9,Rising interest rates weigh on Tessaro Real Es...,macroeconomics,negative


### Note for task C
Task C mentions using a "browser-based LLM", and i interpret that as using Gemini through the *Python API*. I Interpreted this as the correct thing asked for in Task C, as the question is slightly vague. I chose this because the course has focused on connecting Python to LLMs, and using the API keeps the comparison inside the notebook.

In [212]:
# Task C

from dotenv import load_dotenv
import os
from google import genai

load_dotenv()

api_key = os.environ.get("API_KEY") #Please note if you run this code, that the api key is named API_KEY
gemini_client = genai.Client(api_key=api_key)

def ask_llm(dataset, model):
    prompt = f"""
Read through the news headlines and categorize them according to:

topic: earnings, mergers, regulation, macroeconomics
sentiment: positive, negative, neutral

Return only a JSON list.
Do not include headlines.
Do not use markdown.
Do not use ```json.
Do not include explanations.

News = {dataset}
"""

    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt
    )

    return response.text

In [213]:
# Task C Gemini result

summary_gemini = pd.read_json(StringIO(ask_llm(news, "gemini-2.5-flash")))
display(summary_gemini)

,topic,sentiment
0,earnings,positive
1,earnings,negative
2,earnings,neutral
3,mergers,neutral
4,mergers,negative
5,mergers,neutral
6,regulation,negative
7,regulation,positive
8,macroeconomics,positive
9,macroeconomics,negative


In [214]:
# Task C Display headlines for easier comparison below

display(news["headline"])

0    Nordion Industries beats Q1 earnings estimates...
1    Helvora Pharmaceuticals misses earnings foreca...
2    Aurelis Bank reports steady quarterly profit, ...
3    Veridyne Logistics to acquire rival Trantec in...
4    Antitrust regulators block proposed merger bet...
5    Kestrel Semiconductor confirms early-stage mer...
6    New EU AI Act compliance rules expected to rai...
7    Finnish FSA grants Norvik Capital expanded lic...
8    Eurozone inflation cools to 2.1%, easing press...
9    Rising interest rates weigh on Tessaro Real Es...
Name: headline, dtype: object

#### NOTE
LLM outputs can vary slightly between runs. Therefore, the tables below are saved example outputs from one run, used as references for the comparison. If the cells above are rerun, the exact model answers may change slightly, but the comparison logic remains the same.

#### This is what Gemini 2.5 flash answered to headlines above

| topic          | sentiment   |
|:---------------|:------------|
| earnings       | positive    |
| earnings       | negative    |
| earnings       | neutral     |
| mergers        | neutral     |
| regulation     | negative    |
| mergers        | neutral     |
| regulation     | negative    |
| regulation     | positive    |
| macroeconomics | positive    |
| macroeconomics | negative    |







#### This is what Gemma3:4b answered to headliens above

| topic          | sentiment   |
|:---------------|:------------|
| earnings       | positive    |
| earnings       | negative    |
| earnings       | positive    |
| mergers        | positive    |
| mergers        | negative    |
| mergers        | neutral     |
| regulation     | neutral     |
| regulation     | positive    |
| macroeconomics | positive    |
| macroeconomics | negative    |

### Summary

Gemini 2.5 Flash is expected to be more capable than the locally run `gemma3:4b` model. Overall, the answers were fairly similar, even after running the code multiple times. The main difference was that one model sometimes interpreted a headline as more neutral than the other.

However, there were no major contradictions where one model classified the same headline as clearly positive and the other as clearly negative. Based on this, Gemini 2.5 Flash seems slightly more reliable, but `gemma3:4b` also performed well for this task.

### Task 3: Supervised machine learning (5p)

For this task, use a subset of the [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) dataset, by downloading and importing the `bank-additional.csv` from the UCI repository. You can find a description of the dataset behind the link.

The goal is to predict whether a prospective customer will subscribe to a term deposit (variable y).

a) Import the dataset and conduct exploratory data analysis on it. (1p)

b) Preprocess the data using the appropriate methods as described in the course materials. (1p)

c) Determine whether this is a classification or regression task. Choose three different machine learning algorithms from scikit-learn and explain briefly why you chose them. For each of the selected algorithsm, train and a model and iteratively adjust the hyperparameters until you no longer manage to improve the performance. (1p)

d) Compare using train, validation and test set split versus using cross-validation. Which one performs better? (1p)

e) Report and evaluate the performance of the models using several of the metrics provided in the course, and explain which model is the best for the task and why. (1p)


In [215]:
# Task A

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/bank-additional.csv", sep=";")
print("Shape:", df.shape)

display(df.head())
display(df.info())
display(df.describe())

print("checking target variable distribution")
display(df["y"].value_counts())

Shape: (4119, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   object 
 2   marital         4119 non-null   object 
 3   education       4119 non-null   object 
 4   default         4119 non-null   object 
 5   housing         4119 non-null   object 
 6   loan            4119 non-null   object 
 7   contact         4119 non-null   object 
 8   month           4119 non-null   object 
 9   day_of_week     4119 non-null   object 
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   object 
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   f

None

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


checking target variable distribution


y
no     3668
yes     451
Name: count, dtype: int64

In [216]:
# Task B

from sklearn.preprocessing import StandardScaler

df_encoded = pd.get_dummies(df, drop_first = True, dtype = int)

std_scalers = [
    "age",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed"
]

df_encoded[std_scalers] = StandardScaler().fit_transform(df_encoded[std_scalers])

X = df_encoded.drop(columns = "y_yes")
y = df_encoded["y_yes"]

print(X.shape)
print(y.shape)



(4119, 53)
(4119,)


### Summary for task C
This is a classification task, because the target variable y_yes only has two possible outcomes: the customer subscribed or did not subscribe.

I chose SVM, Random Forest, and KNN because they are three different classification models. SVM works well with scaled data, Random Forest is good for tabular data and non-linear patterns, and KNN is a simple model based on distances between observations.

For each model, I first trained a basic version. After that, I adjusted the main hyperparameters with cross-validation and compared the results on the validation set.

In [217]:
# Task C

# This is a classification task
# The classification models used are: 1. Random Forest, 2. SVM, 3. KNN

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold, cross_val_score


X_temp, X_test, y_temp, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.25, random_state = 42)

In [218]:
# Task C, building the svm model

svm_model = SVC(class_weight = "balanced", random_state = 42)

# Training on the training data
svm_model.fit(X_train, y_train)

# Prediction the validation set data
svm_prediction = svm_model.predict(X_val)

print("Original SVM")
print(classification_report(y_val, svm_prediction))
print(confusion_matrix(y_val, svm_prediction))



# Adjusting the model hyperparameters

param_grid_svm = {
    "C" : [0.1,1,10,100],
    "gamma" : ["scale",0.01,0.1,1,0.001,0.0001],
    "kernel" : ["rbf"]
}

grid_svm = GridSearchCV(SVC(random_state = 42, class_weight = "balanced"), param_grid_svm, cv=5, scoring="recall", n_jobs=-1)
grid_svm.fit(X_train,y_train)

print("Adjusted SVM")
print("Best SVM parameters:", grid_svm.best_params_)
print("Best SVM Recall (CV):", f"{grid_svm.best_score_:.4f}")

# New adjusted prediction the validation set data
svm_prediction_adjusted = grid_svm.best_estimator_.predict(X_val)


print(classification_report(y_val, svm_prediction_adjusted))
print(confusion_matrix(y_val, svm_prediction_adjusted))


Original SVM
              precision    recall  f1-score   support

           0       0.98      0.86      0.92       730
           1       0.44      0.87      0.59        94

    accuracy                           0.86       824
   macro avg       0.71      0.87      0.75       824
weighted avg       0.92      0.86      0.88       824

[[627 103]
 [ 12  82]]
Adjusted SVM
Best SVM parameters: {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
Best SVM Recall (CV): 0.8792
              precision    recall  f1-score   support

           0       0.98      0.85      0.91       730
           1       0.42      0.88      0.57        94

    accuracy                           0.85       824
   macro avg       0.70      0.86      0.74       824
weighted avg       0.92      0.85      0.87       824

[[617 113]
 [ 11  83]]


In [219]:
# Task C, building the RandomForest model

rf_model = RandomForestClassifier(class_weight="balanced",random_state = 42)

# Training on the training data
rf_model.fit(X_train, y_train)

# Prediction the validation set data
rf_prediction = rf_model.predict(X_val)

print("Original RF")
print(classification_report(y_val,rf_prediction))
print(confusion_matrix(y_val,rf_prediction))


# Adjusting the model hyperparameters

param_grid_rf ={
    "n_estimators": [50,100,200,300],
    "max_depth": [5,10,20,30,None],
    "bootstrap": [True, False]
}

grid_rf = GridSearchCV(RandomForestClassifier(class_weight="balanced",random_state = 42), param_grid_rf, cv=5, scoring="recall", n_jobs=-1)
grid_rf.fit(X_train,y_train)

print("Adjusted RF")
print("Best RF parameters:", grid_rf.best_params_)
print("Best RF Recall (CV):", f"{grid_rf.best_score_:.4f}")

rf_prediction_adjusted = grid_rf.best_estimator_.predict(X_val)

print(classification_report(y_val, rf_prediction_adjusted))
print(confusion_matrix(y_val, rf_prediction_adjusted))


Original RF
              precision    recall  f1-score   support

           0       0.91      0.98      0.95       730
           1       0.68      0.28      0.39        94

    accuracy                           0.90       824
   macro avg       0.80      0.63      0.67       824
weighted avg       0.89      0.90      0.88       824

[[718  12]
 [ 68  26]]
Adjusted RF
Best RF parameters: {'bootstrap': False, 'max_depth': 5, 'n_estimators': 100}
Best RF Recall (CV): 0.8415
              precision    recall  f1-score   support

           0       0.98      0.86      0.92       730
           1       0.45      0.87      0.60        94

    accuracy                           0.87       824
   macro avg       0.72      0.87      0.76       824
weighted avg       0.92      0.87      0.88       824

[[631  99]
 [ 12  82]]


In [220]:
# Task C, building the KNN model

knn_model = KNeighborsClassifier(n_neighbors = 5)

# Training on the training data
knn_model.fit(X_train, y_train)

# Prediction the validation set data
knn_prediction = knn_model.predict(X_val)

print("Original KNN")
print(classification_report(y_val,knn_prediction))
print(confusion_matrix(y_val,knn_prediction))


# Adjusting the model hyperparameters

neighbors = list(range(3, 50, 2))
cv_scores = []

for k in neighbors:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train, y_train, cv=10, scoring="recall")
    cv_scores.append(scores.mean())

best_index = cv_scores.index(max(cv_scores))
best_k = neighbors[best_index]

print("Adjusted KNN")
print("Best k:", best_k)
print("Best CV recall:", max(cv_scores))

knn_model_adjusted = KNeighborsClassifier(n_neighbors = best_k)
knn_model_adjusted.fit(X_train,y_train)

knn_prediction_adjusted = knn_model_adjusted.predict(X_val)
print(classification_report(y_val, knn_prediction_adjusted))
print(confusion_matrix(y_val, knn_prediction_adjusted))


Original KNN
              precision    recall  f1-score   support

           0       0.92      0.98      0.95       730
           1       0.67      0.32      0.43        94

    accuracy                           0.90       824
   macro avg       0.79      0.65      0.69       824
weighted avg       0.89      0.90      0.89       824

[[715  15]
 [ 64  30]]
Adjusted KNN
Best k: 3
Best CV recall: 0.38774928774928774
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       730
           1       0.61      0.40      0.49        94

    accuracy                           0.90       824
   macro avg       0.77      0.69      0.72       824
weighted avg       0.89      0.90      0.89       824

[[706  24]
 [ 56  38]]


In [221]:
# Task D
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import recall_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

svm_crossval = cross_val_score(grid_svm.best_estimator_, X_train, y_train, cv=kf, scoring="recall")
rf_crossval = cross_val_score(grid_rf.best_estimator_, X_train, y_train, cv=kf, scoring="recall")
knn_crossval = cross_val_score(knn_model_adjusted, X_train, y_train, cv=kf, scoring="recall")


comparison = pd.DataFrame({
    "Model": ["SVM", "Random Forest", "KNN"],
    "Validation recall": [
        recall_score(y_val, svm_prediction_adjusted),
        recall_score(y_val, rf_prediction_adjusted),
        recall_score(y_val, knn_prediction_adjusted)
    ],
    "Cross-validation recall": [
        svm_crossval.mean(),
        rf_crossval.mean(),
        knn_crossval.mean()
    ]
})

display(comparison)


,Model,Validation recall,Cross-validation recall
0,SVM,0.882979,0.884924
1,Random Forest,0.872340,0.814979
2,KNN,0.404255,0.354548


### Summary for Task D

The validation split and cross-validation gave very similar results for SVM, which makes the SVM result more trustworthy. Random Forest had a higher validation recall than cross-validation recall, which suggests that the single validation split may have been slightly optimistic. KNN performed weakly in both approaches.

Overall, cross-validation is more reliable because it evaluates the model across several different splits instead of depending on only one validation set. Based on both validation recall and cross-validation recall, SVM performs best.

In [222]:
# Task E

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

svm_test_prediction = grid_svm.best_estimator_.predict(X_test)
rf_test_prediction = grid_rf.best_estimator_.predict(X_test)
knn_test_prediction = knn_model_adjusted.predict(X_test)

results = pd.DataFrame({
    "Model": ["SVM", "RF", "KNN"],
    "Accuracy": [accuracy_score(y_test, svm_test_prediction),
                 accuracy_score(y_test, rf_test_prediction),
                 accuracy_score(y_test, knn_test_prediction)],
    "Precision": [precision_score(y_test, svm_test_prediction),
                  precision_score(y_test, rf_test_prediction),
                  precision_score(y_test, knn_test_prediction)],
    "Recall": [recall_score(y_test, svm_test_prediction),
               recall_score(y_test, rf_test_prediction),
               recall_score(y_test, knn_test_prediction)],
    "f1 score": [f1_score(y_test, svm_test_prediction),
                 f1_score(y_test, rf_test_prediction),
                 f1_score(y_test, knn_test_prediction)]
})


display(results)

print("SVM confusion matrix:")
print(confusion_matrix(y_test, svm_test_prediction))

print("Random Forest confusion matrix:")
print(confusion_matrix(y_test, rf_test_prediction))

print("KNN confusion matrix:")
print(confusion_matrix(y_test, knn_test_prediction))

,Model,Accuracy,Precision,Recall,f1 score
0,SVM,0.836165,0.390863,0.836957,0.532872
1,RF,0.847087,0.402299,0.760870,0.526316
2,KNN,0.893204,0.531250,0.369565,0.435897


SVM confusion matrix:
[[612 120]
 [ 15  77]]
Random Forest confusion matrix:
[[628 104]
 [ 22  70]]
KNN confusion matrix:
[[702  30]
 [ 58  34]]


### Summary

The models were evaluated on the test set using accuracy, precision, recall, F1-score, and confusion matrices. Since the target variable is imbalanced, accuracy alone is not enough. Recall and F1-score are more useful here because class 1 represents customers who subscribed to the term deposit.

Based on the test results, SVM was the best model for this task. It had the strongest recall for class 1, meaning it identified the largest share of actual subscribers. This is important because the goal is to find potential customers who are likely to subscribe, not only to predict the majority class correctly.

### Statement of AI use

AI was used to help with debugging code and improving some markdown explanations. AI was also used to help phrase the summaries more clearly. The code and outputs were created, checked and adapted by me.